# Representative-Operator Equivalence: PyTorch vs. NCET

This experiment builds one compact neural network containing representative affine, pooling, activation, branching, shape, and indexing operators. After fixing an input $x^*$, it checks that the NCET formulation permits only the output $f_{NN}(x^*)$ computed directly by PyTorch. NCET bounds and optimization expressions describe one sample and therefore omit the PyTorch batch dimension.

For each output element $y_j$, solve both

$$\underline y_j=\min y_j, \qquad \overline y_j=\max y_j.$$

If the encoding is exact, then

$$\underline y_j=\overline y_j=f_{NN}(x^*)_j.$$

In [ ]:
from __future__ import annotations

import cvxpy as cp
import numpy as np
import torch
from torch import nn

from ncet import Bounds, form_milp

torch.set_default_dtype(torch.float64)
np.set_printoptions(precision=7, suppress=True)

## 1. Build a Small Representative Network

The network uses the following execution path:

```text
Input -> Conv2d -> ReLU -> AvgPool2d ----------- Add ---- Sub --+
                         |                       ^        ^       |
                         +-> Slice shortcut -----+        |       |
                         +-> MaxPool2d -------------------+       |
                                                                  v
                  Concat -> Permute -> Flatten -> Linear -> ReLU -> Linear
```

This covers representative operators from every major supported category without trying to include every similar shape operation or indexing spelling. The per-sample input domain is $x\in[-1,1]^{1\times4\times4}$. It also includes several residual connections.

In [ ]:
class RepresentativeOperatorNetwork(nn.Module):
    # The NN parameters are given arbitrarily
    def __init__(self) -> None:
        super().__init__()
        self.conv = nn.Conv2d(1, 1, kernel_size=3, padding=1)
        self.avg_pool = nn.AvgPool2d(kernel_size=2)
        self.max_pool = nn.MaxPool2d(kernel_size=2)
        self.flatten = nn.Flatten(start_dim=1)
        self.hidden = nn.Linear(8, 3)
        self.output = nn.Linear(3, 2)

        with torch.no_grad():
            self.conv.weight.copy_(
                torch.tensor(
                    [
                        [0.3, -0.2, 0.1],
                        [0.4, 0.2, -0.3],
                        [-0.1, 0.25, 0.15],
                    ]
                ).reshape_as(self.conv.weight)
            )
            self.conv.bias.fill_(-0.05)
            self.hidden.weight.copy_(
                torch.linspace(-0.4, 0.5, self.hidden.weight.numel()).reshape_as(
                    self.hidden.weight
                )
            )
            self.hidden.bias.copy_(torch.tensor([0.0, 1.0, -1.0]))
            self.output.weight.copy_(
                torch.tensor(
                    [
                        [0.5, -0.2, 0.3],
                        [-0.4, 0.6, -0.1],
                    ]
                )
            )
            self.output.bias.copy_(torch.tensor([0.1, -0.2]))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = torch.relu(self.conv(x))
        average = self.avg_pool(features)
        maximum = self.max_pool(features)
        shortcut = features[:, :, ::2, ::2]  # Slice shortcut
        residual = average + shortcut        # Add residual
        contrast = residual - maximum        # Subtract max-pooled
        merged = torch.cat((residual, contrast), dim=1)  # Concatenate
        reordered = merged.permute(0, 2, 3, 1)  # Permute dimensions
        hidden = torch.relu(self.hidden(self.flatten(reordered)))  # ReLU hidden
        return self.output(hidden)


model = RepresentativeOperatorNetwork().eval()
input_bounds = Bounds(
    lower=np.full((1, 4, 4), -1.0),
    upper=np.full((1, 4, 4), 1.0),
) # # alternatively, input bounds can be provided as a tuple (lower, upper), list of tuples for multiple inputs, and dictionary {x: (lower, upper)}

model

## 2. Build the Reduced and Full Formulations

Both modes are exact. Reduced mode creates binary variables only for unstable ReLU elements, whereas full mode creates one for every ReLU element. MaxPool2d uses its full exact selector formulation in either mode.

In [ ]:
encodings = {
    mode: form_milp(
        model=model,
        input_bounds=input_bounds,
        relu_binary_mode=mode,
    )
    for mode in ("reduced", "full")
}

operators = [
    node.op_type
    for node in encodings["reduced"].graph.nodes
    if node.op_type not in {"Input", "Output"}
]
print("canonical operators:", " -> ".join(operators))

for mode, encoding in encodings.items():
    print(
        f"{mode:7s}: continuous={encoding.stats.continuous_variables}, "
        f"binary={encoding.stats.binary_variables}, "
        f"active={encoding.stats.always_active_relu}, "
        f"inactive={encoding.stats.always_inactive_relu}, "
        f"unstable={encoding.stats.unstable_relu}"
    )

## 3. Fix the Input and Compute the MILP Output Interval

`encoding.outputs[0]` is the CVXPY variable representing the model output. After fixing the input, minimize and maximize each scalar output separately.

In [ ]:
def pytorch_output(sample: np.ndarray) -> np.ndarray:
    tensor = torch.from_numpy(sample).to(dtype=torch.get_default_dtype())
    with torch.no_grad():
        return model(tensor.unsqueeze(0)).squeeze(0).numpy()


def milp_output_interval(
    encoding,
    sample: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    
    output = encoding.outputs[0]     # Optimization target
    fixed_constraints = [
        *encoding.constraints,
        encoding.inputs["x"] == sample,  # Fix the input
    ]

    lower = np.empty(output.size)
    upper = np.empty(output.size)
    valid_statuses = {cp.OPTIMAL, cp.OPTIMAL_INACCURATE}

    for index in range(output.size):
        # For each output element, solve the MILP by minimizing and maximizing separately
        minimum = cp.Problem(cp.Minimize(output[index]), fixed_constraints)
        maximum = cp.Problem(cp.Maximize(output[index]), fixed_constraints)
        minimum.solve(solver=cp.SCIPY)
        maximum.solve(solver=cp.SCIPY)

        if minimum.status not in valid_statuses or maximum.status not in valid_statuses:
            raise RuntimeError(
                f"unexpected solver status: {minimum.status}, {maximum.status}"
            )

        lower[index] = minimum.value
        upper[index] = maximum.value

    return lower.reshape(output.shape), upper.reshape(output.shape)

## 4. Run the Pointwise Experiment

The test points include a zero image, a smooth ramp, and a checkerboard pattern. The success criteria are:

- the maximum error between the PyTorch output and the MILP minimum/maximum is no greater than the tolerance;
- the MILP output-interval width (maximum - minimum) after fixing the input is no greater than the tolerance;
- both reduced and full modes satisfy these conditions.

In [ ]:
samples = [
    np.zeros((1, 4, 4)),
    np.linspace(-1.0, 1.0, 16).reshape(1, 4, 4),
    np.array(
        [
            [
                [1.0, -1.0, 1.0, -1.0],
                [-1.0, 1.0, -1.0, 1.0],
                [1.0, -1.0, 1.0, -1.0],
                [-1.0, 1.0, -1.0, 1.0],
            ]
        ]
    ),
]

tolerance = 1e-6
results = []

for mode, encoding in encodings.items():
    for sample_index, sample in enumerate(samples):
        direct = pytorch_output(sample)
        lower, upper = milp_output_interval(encoding, sample)
        output_error = max(
            np.max(np.abs(lower - direct)),
            np.max(np.abs(upper - direct)),
        )
        interval_width = np.max(upper - lower)
        results.append((mode, sample_index, output_error, interval_width))

        print(
            f"{mode:7s} sample={sample_index} torch={direct.ravel()} "
            f"milp=[{lower.ravel()}, {upper.ravel()}]"
        )

max_output_error = max(result[2] for result in results)
max_interval_width = max(result[3] for result in results)

assert max_output_error <= tolerance
assert max_interval_width <= tolerance

print(f"\nmax output error:   {max_output_error:.3e}")
print(f"max interval width: {max_interval_width:.3e}")
print("Pointwise equivalence checks passed.")

## 5. Interpret the Results

For every test input, agreement among the MILP minimum, MILP maximum, and PyTorch output shows that the combined representative-operator graph contains the correct output and introduces no additional output at that fixed input.